In [1]:
import requests
from bs4 import BeautifulSoup
import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import plotly.express as px
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import pandas_datareader.data as web

import warnings
warnings.filterwarnings('ignore')

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, "lxml")

# Find the components table (wikitable sortable)
table = soup.find("table", {"class": "wikitable sortable"})
if table is None:
    raise ValueError("Could not locate the components table.")

tickers = []

# All exchange prefixes to remove
prefixes = [
    "NYSE:", "Nasdaq:", "FWB:", "LSE:", "SWX:", "HKEX:", "TSX:",
    "BME:", "EPA:", "BIT:", "TSE:", "ASX:", "KRX:", "SIX:",
    "Euronext:", "OTC:", "JSE:", "BMV:", "SGX:", "NSE:", "BSE:"
]

prefix_clean = [p.replace(":", "") for p in prefixes]

# Extract rows
for row in table.find_all("tr")[1:]:  # skip header
    cols = row.find_all("td")
    if len(cols) >= 3:
        raw = cols[2].get_text(" ", strip=True)

        # Remove prefixes
        for p in prefixes:
            raw = raw.replace(p, "")

        # Split multiple tickers: ; , / space
        parts = re.split(r"[;,/ ]+", raw)

        for t in parts:
            t = t.strip().replace(".", "-")

            # Remove garbage entries
            if not t or t in [":", "-", "/", "|"]:
                continue
            if t.isdigit():
                continue

            # Remove index names appearing as tokens
            if t.lower() in [i.lower() for i in prefix_clean]:
                continue

            tickers.append(t)

# Deduplicate and sort
tickers = sorted(set(tickers))

# Build DataFrame (tickers only)
GLOBAL_Titans_TOP50 = pd.DataFrame(tickers, columns=["Ticker"])

# Save CSV
csv_filename = "GLOBAL_Titans_TOP50.csv"
GLOBAL_Titans_TOP50.to_csv(csv_filename, index=False)

print(GLOBAL_Titans_TOP50)
print(f"Saved CSV as: {csv_filename}")


   Ticker
0    AAPL
1    ABBV
2     ABI
3     ALV
4    AMGN
5    AMZN
6      BA
7    BATS
8     BHP
9      BP
10      C
11   CSCO
12    CVX
13     DD
14    DIS
15     FB
16     GE
17   GOOG
18  GOOGL
19    GSK
20   HSBA
21    IBM
22   INTC
23    JNJ
24    JPM
25     KO
26     MA
27    MCD
28    MMM
29    MRK
30   MSFT
31   NESN
32   NOVN
33   NVDA
34   ORCL
35    PEP
36    PFE
37     PG
38     PM
39   RDSA
40    ROG
41     RY
42    SAN
43    SIE
44   SMSN
45    TOT
46    TSM
47    TYO
48      V
49    WMT
50    XOM
Saved CSV as: GLOBAL_Titans_TOP50.csv


In [9]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')

tables = soup.find_all('table')

# The S&P 500 table is the first one
df = pd.read_html(str(tables[0]))[0]

# Extract tickers
tickers = (
    df['Symbol']
    .astype(str)
    .str.replace('.', '-', regex=False)
    .str.strip()
    .tolist()
)

# Remove nan, None, empty strings
clean_tickers = [
    t for t in tickers
    if t.lower() not in ["nan", "none", ""]
]

# Create DataFrame with index name as variable name
index_name = "SP500_Tickers"
sp500_df = pd.DataFrame(clean_tickers, columns=["Ticker"])
sp500_df.index.name = index_name

# Save CSV using the index name
csv_filename = f"{index_name}.csv"
sp500_df.to_csv(csv_filename)

print(sp500_df)
print(f"Saved CSV as: {csv_filename}")


              Ticker
SP500_Tickers       
0                MMM
1                AOS
2                ABT
3               ABBV
4                ACN
...              ...
498              XYL
499              YUM
500             ZBRA
501              ZBH
502              ZTS

[503 rows x 1 columns]
Saved CSV as: SP500_Tickers.csv


In [10]:
url = "https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies"
headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.text, "lxml")

tables = soup.find_all("table")

# Find the table with a Symbol or Ticker column
target_table = None
ticker_column = None

for t in tables:
    df_test = pd.read_html(str(t))[0]
    cols = [str(col).lower() for col in df_test.columns]

    for possible in ["symbol", "ticker", "nasdaq", "company ticker"]:
        if possible in cols:
            ticker_column = df_test.columns[cols.index(possible)]
            target_table = df_test
            break

    if target_table is not None:
        break

if target_table is None:
    raise ValueError("No ticker column found in any table.")

df = target_table

# Extract tickers
tickers = (
    df[ticker_column]
    .astype(str)
    .str.replace(".", "-", regex=False)
    .str.strip()
    .tolist()
)

# Remove nan, None, empty strings
clean_tickers = [
    t for t in tickers
    if t.lower() not in ["nan", "none", ""]
]

# Create DataFrame with index name as variable name
index_name = "NASDAQ100_Tickers"
nasdaq100_df = pd.DataFrame(clean_tickers, columns=["Ticker"])
nasdaq100_df.index.name = index_name

# Save CSV using the index name
csv_filename = f"{index_name}.csv"
nasdaq100_df.to_csv(csv_filename)

print(nasdaq100_df)
print(f"Saved CSV as: {csv_filename}")


                  Ticker
NASDAQ100_Tickers       
0                   ADBE
1                    AMD
2                   ABNB
3                   ALNY
4                  GOOGL
...                  ...
97                   WMT
98                   WBD
99                   WDC
100                 WDAY
101                  XEL

[102 rows x 1 columns]
Saved CSV as: NASDAQ100_Tickers.csv


In [11]:
# Load all four universes
djia = pd.read_csv("GLOBAL_Titans_TOP50.csv")
sp500 = pd.read_csv("SP500_Tickers.csv")
nasdaq100 = pd.read_csv("NASDAQ100_Tickers.csv")
r1000 = pd.read_csv("Russell1000_Tickers.csv")

# Combine into one master list
combined = pd.concat([djia, sp500, nasdaq100, r1000], ignore_index=True)

# Deduplicate and sort
unique_tickers = (
    combined["Ticker"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

# Create final DataFrame named stock_symbols
stock_symbols = pd.DataFrame(unique_tickers, columns=["Ticker"])

# Reset index for clean data
stock_symbols = stock_symbols.reset_index(drop=True)

# Save CSV named stock_symbols.csv
stock_symbols.to_csv("stock_symbols.csv", index=False)

print(stock_symbols)
print("Saved CSV as: stock_symbols.csv")


     Ticker
0         A
1        AA
2       AAL
3      AAOI
4      AAON
...     ...
1042     ZG
1043   ZION
1044     ZM
1045     ZS
1046    ZTS

[1047 rows x 1 columns]
Saved CSV as: stock_symbols.csv


In [12]:
def load_stock_symbols(csv_path="stock_symbols.csv", return_list=False):
    """
    Load the master stock symbol universe from stock_symbols.csv.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the stock symbols.
    return_list : bool
        If True, return a Python list of tickers instead of a DataFrame.

    Returns
    -------
    DataFrame or list
        Cleaned stock symbols.
    """

    # Load CSV
    df = pd.read_csv(csv_path)

    # Clean tickers
    df["Ticker"] = (
        df["Ticker"]
        .astype(str)
        .str.strip()
        .str.replace(".", "-", regex=False)
    )

    # Reset index for cleanliness
    df = df.reset_index(drop=True)

    if return_list:
        return df["Ticker"].tolist()

    return df


In [13]:
stock_symbols = load_stock_symbols()
print(stock_symbols)


     Ticker
0         A
1        AA
2       AAL
3      AAOI
4      AAON
...     ...
1042     ZG
1043   ZION
1044     ZM
1045     ZS
1046    ZTS

[1047 rows x 1 columns]
